In [7]:
# import packages
import os

import pandas as pd

from calendar import monthrange
from datetime import datetime, timedelta

from azureml.core import Dataset, Datastore, Workspace
from azureml.opendatasets import NoaaIsdWeather

In [8]:
ws = Workspace.from_config()
dstore = ws.get_default_datastore()

In [9]:
ws

Workspace.create(name='DataDrift-Prod-EUAP', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG')

In [ ]:
%%time 

target_years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

for year in target_years:
    for month in range(1, 12+1):
        path = 'weather-data/{}/{:02d}/'.format(year, month)
        
        try:  
            start = datetime(year, month, 1)
            end   = datetime(year, month, monthrange(year, month)[1]) + timedelta(days=1)
            isd   = NoaaIsdWeather(start, end).to_pandas_dataframe()
            isd   = isd[isd['stationName'].str.contains('FLORIDA|WASHINGTON|TEXAS|MIAMI|SEATTLE|HOUSTON', regex=True, na=False)]
            
            os.makedirs(path, exist_ok=True)
            isd.to_parquet(path + 'data.parquet')
        except Exception as e:
            print('Month {} in year {} likely has no data.\n'.format(month, year))
            print('Exception: {}'.format(e))

ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2010/month=1/', '/year=2010/month=2/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2010/month=1/part-00009-tid-2760641154746282667-d6a50598-0478-455f-9217-ce4afb96a9bb-36.c000.snappy.parquet under container isdweatherdatacontainer
Reading ISDWeather/year=2010/month=2/part-00011-tid-2760641154746282667-d6a50598-0478-455f-9217-ce4afb96a9bb-38.c000.snappy.parquet under container isdweatherdatacontainer
Done.
ActivityCompleted: Activity=to_pandas_dataframe_in_worker, HowEnded=Success, Duration=101898.4 [ms]
ActivityCompleted: Activity=to_pandas_dataframe, HowEnded=Success, Duration=101922.07 [ms]
ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2010/month=2/', '/year=2010/month=3/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2010/month=

In [ ]:
# uploads everything in the folder 'data' to the datastore under a container named 'weather-data'
dstore.upload('weather-data', 'weather-data')